# 5. Single-User Table Lookup

Once the precomputed table exists, user assignment no longer starts from a fresh single-user solve. It starts from the stored frontier for the appropriate distance bin and keeps only the rows that can serve the user's required active rate.


## 1. Build one small scheduler-facing user batch

To keep the lookup stage concrete, this notebook reuses the synthetic day-user generator and selects one small active bin. The selected rows are already in the same user-table contract consumed by the day runner and the TDMA scheduler.


In [ ]:
import sys
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

for path in (repo_root / "notebooks", repo_root / "src"):
    resolved = str(path.resolve())
    if resolved not in sys.path:
        sys.path.insert(0, resolved)

from configs.day_cycle import (
    DEFAULT_DAY_CYCLE_LOAD_CURVE_CSV,
    DEFAULT_SYNTHETIC_SESSION_GENERATION_CONFIG,
)
from helpers.DayCycleSimulationHelpers import build_day_cycle_discussion_artifacts, bin_index_to_clock
from helpers.table_lookup_helpers import (
    build_table_lookup_artifacts,
    pick_example_scheduler_bin,
    plot_single_user_lookup,
)

load_curve_csv = Path(DEFAULT_DAY_CYCLE_LOAD_CURVE_CSV)
if not load_curve_csv.is_absolute():
    load_curve_csv = repo_root / load_curve_csv

config = DEFAULT_SYNTHETIC_SESSION_GENERATION_CONFIG
day_cycle_artifacts = build_day_cycle_discussion_artifacts(load_curve_csv, config)
example_bin_index = pick_example_scheduler_bin(
    day_cycle_artifacts["scheduler_day_user_table"],
    target_user_count=4,
)
example_user_table = (
    day_cycle_artifacts["scheduler_day_user_table"]
    .loc[
        lambda table: table["bin_index"].eq(example_bin_index),
        ["user_id", "distance_m", "required_rate_bps"],
    ]
    .reset_index(drop=True)
)
lookup_artifacts = build_table_lookup_artifacts(example_user_table)
example_user_id = next(
    user_id
    for user_id, candidate_space in lookup_artifacts.user_candidate_spaces.items()
    if not candidate_space.empty
)
example_user_row = example_user_table.loc[
    example_user_table["user_id"].eq(example_user_id)
].iloc[0]
example_candidate_space = lookup_artifacts.user_candidate_spaces[example_user_id]

print(f"Example bin: {example_bin_index} ({bin_index_to_clock(example_bin_index)})")


In [ ]:
display(example_user_table)
display(lookup_artifacts.assigned_user_table)


## 2. Inspect one user's matched stored frontier

Each user is snapped onto a precomputed distance bin, then filtered by the requested active rate. The remaining rows are the potential allocations that the TDMA layer can later quantize in time.


In [ ]:
display(example_candidate_space.head(12))
plot_single_user_lookup(
    example_candidate_space,
    user_label=f"user {example_user_id} at {float(example_user_row['distance_m']):.0f} m",
    required_rate_bps=float(example_user_row["required_rate_bps"]),
    pa_label_map=lookup_artifacts.pa_label_map,
)


This is the bridge from stored single-user frontiers to the joint scheduler: each user now carries a small candidate set that already reflects the distance bin and the requested active rate. The next notebook uses those per-user lookup tables as the starting point for exact TDMA scheduling.
